### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:

from langchain_groq import ChatGroq

model = ChatGroq(model="qwen/qwen3-32b")

response = model.invoke("Hello, from today your name will be MASH.")
response.content

'<think>\nOkay, the user wants me to call myself MASH from now on. Let me make sure I understand correctly - they\'re giving me a new name, right? I need to check if there are any specific reasons they want to change the name. Maybe it\'s for convenience, or they\'re creating a new scenario? \n\nFirst, I should confirm that I can indeed use the name MASH. The name itself doesn\'t seem to conflict with my system guidelines. But wait, there might be a system name or model number associated with me. However, the user is asking for a nickname, so it\'s probably acceptable as long as I stay within the boundaries of my original functions.\n\nNext, I need to respond to their request. They said, "Hello, from today your name will be MASH." I should acknowledge their request and introduce myself as MASH while also reminding them of my role. Maybe say something like, "Hello, I am now MASH, your helpful AI assistant. How can I assist you today?" That way, I confirm the name change and maintain pro

In [25]:
from datetime import datetime
from langchain.tools import tool

# --- Tool definitions ---
@tool
def get_weather(location: str) -> str:
    """Get the current weather in a given location."""
    return f"The weather in {location} is sunny."

@tool
def get_time(location: str) -> str:
    """Get the current time in a given location."""
    return f"The current time in {location} is 10:30 AM."

@tool
def get_date() -> str:
    """Get today's date."""
    return f"Today's date is {datetime.now().date()}."


# Bind all tools to the model so it knows what tools are available
model_with_tools = model.bind_tools([get_weather, get_time, get_date])


# Step 1 only — ask the model which tool to call (no real answer yet, content will be empty)
# The model reads the question and returns a tool_call request, not a final answer

# --- Call 1: weather question → model should pick get_weather ---
response = model_with_tools.invoke("What is the weather in New York?")
print(response)
print(f"Tool calls: {response.tool_calls}")

# --- Call 2: time question → model should pick get_time ---
response = model_with_tools.invoke("What time is it in Tokyo?")
print(response)
print(f"Tool calls: {response.tool_calls}")

# --- Call 3: date question → model should pick get_date ---
response = model_with_tools.invoke("What is today's date?")
print(response)
print(f"Tool calls: {response.tool_calls}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in New York. Let me check which tool I can use here. The available functions are get_weather, get_time, and get_date. The get_weather function requires a location parameter, which is provided as New York. So I should call get_weather with location set to "New York". The other functions don\'t apply here since the user isn\'t asking for time or date. Just need to make sure the arguments are correctly formatted as JSON within the tool_call tags.\n', 'tool_calls': [{'id': 'e46xsjf8n', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 128, 'prompt_tokens': 253, 'total_tokens': 381, 'completion_time': 0.177511188, 'completion_tokens_details': {'reasoning_tokens': 103}, 'prompt_time': 0.011047675, 'prompt_tokens_details': None, 'queue_time': 0.051414534, 'total_time': 0.188558863}, 'model_name':

### Tool Execution loops

In [26]:
# Step 1: Send user question to the model
# The model does NOT answer directly — it reads the question and decides which tool to call
# The response contains tool_calls (e.g. get_date) but content is empty "" (no real answer yet)
# We append the response to messages so the model remembers it called a tool in Step 3
messages = [{"role": "user", "content": "What's the date?"}]
response = model_with_tools.invoke(messages)
messages.append(response)  # save model's tool decision to message history

# Step 2: We (not the model) actually run the tool the model picked in Step 1
# The model can't run Python functions — it just tells us what to run and with what arguments
# We check which tool was requested, run it, and get the result back
# We append the tool result to messages so the model can see it in Step 3
for tool_call in response.tool_calls:
    tool_name = tool_call["name"]  # e.g. "get_date"

    if tool_name == "get_weather":
        tool_result = get_weather.invoke(tool_call)   # runs get_weather("New York")
    elif tool_name == "get_time":
        tool_result = get_time.invoke(tool_call)       # runs get_time("Tokyo")
    elif tool_name == "get_date":
        tool_result = get_date.invoke(tool_call)       # runs get_date()

    messages.append(tool_result)  # save tool result to message history

# Step 3: Send the full conversation back to the model
# Now messages has: user question + model's tool decision + tool result
# The model reads the tool result and forms a proper final answer for the user
final_response = model_with_tools.invoke(messages)
print(final_response.content)  # fixed: .text → .content

Today's date is **July 6, 2026**.


In [ ]:
# Check what is in messages that we send to the model in Step 3
messages

[{'role': 'user', 'content': "What's the date?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking, "What\'s the date?" I need to figure out which tool to use here. Let me check the available functions. There\'s get_weather, get_time, and get_date. The get_date function doesn\'t require any parameters, and its description says it gets today\'s date. That\'s exactly what the user is asking for. The other functions need a location, which the user hasn\'t provided. Since the user just wants the date, I should use get_date. No arguments needed, so the tool call is straightforward.\n', 'tool_calls': [{'id': '23bkj5v14', 'function': {'arguments': '{}', 'name': 'get_date'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 132, 'prompt_tokens': 250, 'total_tokens': 382, 'completion_time': 0.21868822, 'completion_tokens_details': {'reasoning_tokens': 112}, 'prompt_time': 0.011521124, 'prompt_tokens_details': None, 'queue_time